# CNN accelerator driver

Classifies the course's 10,000 MNIST images on the board and measures the latency as Assignment 4
defines it, from writing the weights to reading the last prediction. The PS only writes the
course's arrays into the accelerator's memories in address order, starts each run and reads the
predictions back; all computation happens in the PL.

1. Copy `cnn.bit` and `cnn.hwh` from `build/bd/cnn/` to the board, next to this notebook and
   `reference/mnist_cnn.npz`.
2. Set SW0 on: it drives the PL's active-low reset.
3. Run the cells in order.

In [ ]:
from pynq import Overlay
import numpy as np
import time

In [ ]:
class CnnDriver:
    BATCH = 3    # images per run

    def __init__(self, bitfile_path):
        self.hw = Overlay(bitfile_path)
        self.csr = self.hw.csrr_0.mmio.array
        self.input = self.hw.SRAM_INPUT.mmio.array
        self.conv_weight = self.hw.SRAM_CONV_WEIGHT.mmio.array
        self.fc_weight = [getattr(self.hw, f"SRAM_FCL_WEIGHT_{k}").mmio.array for k in range(10)]

    def load_weights(self, layer1, layer2, fc1):
        # each memory word holds one signed byte in its low 8 bits
        conv = np.concatenate([layer1.reshape(-1), layer2.reshape(-1)])
        self.conv_weight[0:conv.size] = conv.astype(np.uint8)
        for k in range(10):
            self.fc_weight[k][0:fc1.shape[1]] = fc1[k].astype(np.uint8)

    def classify(self, images):
        # one run: three images of shape (1, 28, 28) in, three predicted digits out
        self.input[0:images.size] = images.reshape(-1).astype(np.uint8)
        self.csr[1] = 1                  # start (the register IP clears the bit again)
        while self.csr[0] != 1:          # wait for done
            pass
        return [int(self.csr[3 + i]) for i in range(self.BATCH)]

In [ ]:
data = np.load("mnist_cnn.npz")
images, reference = data["input"], data["output"]
cnn = CnnDriver("cnn.bit")

In [ ]:
predictions = []
start_time = time.time()
cnn.load_weights(data["layer1"], data["layer2"], data["fc1"])
for first in range(0, len(images), CnnDriver.BATCH):
    batch = images[first:first + CnnDriver.BATCH]
    count = len(batch)
    if count < CnnDriver.BATCH:      # the last run has one image; blank images fill the batch
        blank = np.zeros((CnnDriver.BATCH - count,) + batch.shape[1:], batch.dtype)
        batch = np.concatenate([batch, blank])
    predictions += cnn.classify(batch)[:count]
latency = time.time() - start_time
predictions = np.array(predictions)
print(f"Latency for {len(images)} images: {latency:.3f} s ({latency / len(images) * 1e6:.1f} us per image)")

In [ ]:
expected = reference.argmax(axis=1)    # the lowest class among equal logits, as in the hardware
matches = int((predictions == expected).sum())
print(f"{matches} of {len(images)} predictions match the course's reference logits")